# Route F: redox in the embryonic salivary gland, stages 10-12 against 13-16

The nuclear HyPer7 signal in salivary glands is reported at stage 14. This notebook asks what changes in that tissue across exactly that window, using the embryo atlas of Peng, Jackson et al. 2024 (*Development*), which covers both windows in one study.

Every change is measured **relative to all other embryonic cells**, so anything happening across the whole embryo is removed. Choose **Runtime → Run all**.

In [ ]:
#@title Cell 1 - Settings and connect Google Drive
import gzip, os, re, subprocess, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except ImportError:
    display = print
pd.set_option("display.width", 230); pd.set_option("display.max_columns", None); pd.set_option("display.max_rows", 220)

#@markdown **GSE234602** (Peng, Jackson et al. 2024, *Development*): single-cell RNA-seq of the *Drosophila*
#@markdown embryo across two windows, **stages 10-12** and **stages 13-16**, with the salivary gland annotated.
#@markdown Both windows come from the same study, so the early-to-late comparison carries no batch confound.
#@markdown The HyPer7 nuclear signal in salivary glands is reported at stage 14, in the late window.
GSE = "GSE234602"  #@param {type:"string"}
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/route_f_embryo_sg"  #@param {type:"string"}
#@markdown Which annotated cell type counts as salivary gland (a regular expression, case-insensitive).
TISSUE_PATTERN = "saliv"  #@param {type:"string"}
#@markdown The gene annotation converts FlyBase IDs to symbols. It is the same file Route B and Route D use.
GTF_PATH = "/content/drive/MyDrive/Colab Notebooks/route_b_fly/Drosophila_melanogaster.gtf.gz"  #@param {type:"string"}
GTF_PATH = os.environ.get("ROUTE_F_GTF", GTF_PATH)
N_RANDOM_SETS = 2000  #@param {type:"integer"}
SEED = 1  #@param {type:"integer"}

DATA_DIR = os.environ.get("ROUTE_F_DATA", DATA_DIR)
if DATA_DIR.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
GDIR = os.path.join(DATA_DIR, GSE)
os.makedirs(GDIR, exist_ok=True)
rng = np.random.default_rng(SEED)


class StopHere(Exception):
    """Stops the notebook with a plain-language message."""


print("Dataset:", GSE, "| folder:", GDIR)
print("Salivary gland cells will be those whose annotated cell type matches:", repr(TISSUE_PATTERN))

In [ ]:
#@title Cell 2 - Download the matrix and the cell annotations
EBI = os.environ.get("ROUTE_F_EBI", "https://ftp.ebi.ac.uk/pub/databases/microarray/data/atlas/sc_experiments")
BASE = os.environ.get("ROUTE_F_GEO", "https://ftp.ncbi.nlm.nih.gov/geo")


def fetch(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        print(f"   already here: {os.path.basename(dest)} ({os.path.getsize(dest) / 1e6:.0f} MB)"); return True
    r = subprocess.run(["curl", "-fsSL", "-m", "7200", "-o", dest + ".part", url], capture_output=True, text=True)
    if r.returncode != 0:
        if os.path.exists(dest + ".part"):
            os.remove(dest + ".part")
        return False
    os.replace(dest + ".part", dest)
    print(f"   downloaded {os.path.basename(dest)} ({os.path.getsize(dest) / 1e6:.0f} MB)"); return True


def listing(url, tries=3):
    for _ in range(tries):
        r = subprocess.run(["curl", "-fsSL", "-m", "180", url], capture_output=True, text=True)
        if r.returncode == 0:
            return [h for h in re.findall(r'href="([^"?/][^"]*)"', r.stdout) if not h.startswith("http")]
    return []


# EBI's Single Cell Expression Atlas holds the ready-made matrix and the per-cell annotations.
# GEO carries only raw sequencing for this study, which is far too large for Colab.
acc = "E-GEOD-" + re.sub(r"\D", "", GSE)
d = f"{EBI}/{acc}/"
names = listing(d)
want = [n for n in names if re.search(r"\.mtx$|\.mtx_rows$|\.mtx_cols$", n)]
# prefer the filtered counts over raw, and take one matrix only
mtx = [n for n in want if n.endswith(".mtx")]
pick = next((m for m in mtx if "filtered" in m), mtx[0] if mtx else None)
want = [n for n in want if pick and n.startswith(pick.rsplit(".mtx", 1)[0])]
design = [n for n in names if n.startswith("ExpDesign") or "cell_metadata" in n or n.endswith("condensed-sdrf.tsv")]
found = [(d + n, n) for n in want + design]
if not found:
    raise StopHere(f"Nothing downloadable found at {d}. Open that address in your browser: if it is empty, the "
                   f"study may be listed under a different accession. Send me what you see.")
print(f"Files to download ({len(found)}):")
for _, n in found:
    print("   ", n)
got = {n: os.path.join(GDIR, n) for u, n in found if fetch(u, os.path.join(GDIR, n))}
if not got:
    got = {n: os.path.join(GDIR, n) for n in sorted(os.listdir(GDIR)) if not n.endswith(".part")}
    if got:
        print("Download failed; using files already in your folder:", list(got))
need = [k for k in ("mtx", "mtx_rows", "mtx_cols") if not any(n.endswith("." + k) for n in got)]
if need or not any(n.startswith("ExpDesign") or "cell_metadata" in n or "sdrf" in n for n in got):
    raise StopHere(f"Missing {need or 'the cell annotation table'}. Check your folder {GDIR} and run again.")
MTX = next(p for n, p in got.items() if n.endswith(".mtx"))
ROWS = next(p for n, p in got.items() if n.endswith(".mtx_rows"))
COLS = next(p for n, p in got.items() if n.endswith(".mtx_cols"))
META = next(p for n, p in got.items() if n.startswith("ExpDesign") or "cell_metadata" in n or "sdrf" in n)
print("\nUsing matrix:", os.path.basename(MTX), "| annotations:", os.path.basename(META))

In [ ]:
#@title Cell 3 - Find the salivary gland cells and the two stage windows
meta = pd.read_csv(META, sep="\t", dtype=str, low_memory=False)
meta.columns = [c.strip() for c in meta.columns]
print(f"Annotation table: {len(meta):,} rows x {meta.shape[1]} columns")


#@markdown Leave these empty to find the columns automatically. If that fails, the cell prints every column
#@markdown name; copy the right ones in here and run the cell again.
CELL_TYPE_COLUMN = ""  #@param {type:"string"}
STAGE_COLUMN = ""  #@param {type:"string"}

# Column names arrive differently depending on where the file came from: "Sample Characteristic[cell type]",
# "inferred_cell_type_-_authors_labels", and so on. Match against a flattened version of each name so that
# spaces, underscores, hyphens and brackets all read the same.
flat = {c: re.sub(r"[^a-z0-9]+", " ", c.lower()).strip() for c in meta.columns}


def pick_column(patterns, exclude=()):
    """Find the column whose name matches, preferring annotated values over ontology URLs."""
    for pat in patterns:
        hits = [c for c in meta.columns
                if re.search(pat, flat[c]) and "ontology" not in flat[c]
                and not any(re.search(x, flat[c]) for x in exclude)]
        hits = [c for c in hits if meta[c].notna().any() and meta[c].nunique() > 1]
        if hits:
            return sorted(hits, key=lambda c: meta[c].nunique())[-1]
    return None


cell_col = CELL_TYPE_COLUMN.strip() or pick_column(
    [r"inferred cell type.*author", r"inferred cell type", r"\bcell type\b", r"annotat", r"cluster"])
stage_col = STAGE_COLUMN.strip() or pick_column(
    [r"developmental stage", r"\bstage\b", r"\bage\b", r"time ?point", r"\bhours?\b"])
id_col = next((c for c in meta.columns if re.search(r"^assay$|^id$|cell id|barcode|^run$", flat[c])), meta.columns[0])
for nm, c in (("CELL_TYPE_COLUMN", cell_col), ("STAGE_COLUMN", stage_col)):
    if c and c not in meta.columns:
        raise StopHere(f"{nm} is set to {c!r}, which is not a column in this table.")
print(f"Cell-type column: {cell_col!r}\nStage column:     {stage_col!r}\nCell id column:   {id_col!r}")
if cell_col is None or stage_col is None:
    print("\nColumns available:"); print("\n".join("   " + c for c in meta.columns))
    raise StopHere("Could not identify the cell-type or stage column. Send me the list above and I will set them by name.")

print(f"\nCell types annotated ({meta[cell_col].nunique()}):")
display(meta[cell_col].value_counts().head(40))
print(f"\nStage values ({meta[stage_col].nunique()}):")
display(meta[stage_col].value_counts())


# Some tables give hours after egg laying rather than stages. At 25 C the boundary between stage 12 and
# stage 13 falls at about 9h20, so that is where the two windows are cut.
IN_HOURS = bool(re.search(r"\bhours?\b|\bh ael\b|\btime\b", flat.get(stage_col, "")) or
                meta[stage_col].astype(str).str.contains(r"\bh\b|hour", case=False, na=False).mean() > 0.5)
if IN_HOURS:
    print("\nThe stage column looks like hours after egg laying, so the windows are cut at 9h20 "
          "(the stage 12 to stage 13 boundary at 25 C).")


def window_of(v):
    """Stages 10-12 are the early window, 13-16 the late one. Accepts 'stage 13-16', 'stage 14' or hours."""
    nums = [float(n) for n in re.findall(r"\d+(?:\.\d+)?", str(v))]
    if IN_HOURS:
        if not nums:
            return None
        mid = sum(nums) / len(nums)
        return "early (stages 10-12)" if mid < 9.33 else "late (stages 13-16)"
    nums = [int(n) for n in nums if 1 <= n <= 17]
    if not nums:
        return None
    lo, hi = min(nums), max(nums)
    if hi <= 12:
        return "early (stages 10-12)"
    if lo >= 13:
        return "late (stages 13-16)"
    return None                      # spans both windows: cannot be assigned


meta["window"] = meta[stage_col].map(window_of)
if meta.window.notna().sum() == 0:
    print("\nStage values seen:", list(meta[stage_col].dropna().unique())[:25])
    raise StopHere("No cell could be assigned to an early or late window from the stage column. The values are "
                   "printed above; send them to me and I will set the rule for this table.")
meta["is_sg"] = meta[cell_col].astype(str).str.contains(TISSUE_PATTERN, case=False, na=False)
meta["tissue"] = np.where(meta.is_sg, "salivary gland", "other embryonic cells")
print("\nCells per group:")
display(pd.crosstab(meta.tissue, meta.window))
if not meta.is_sg.any():
    raise StopHere(f"No cell type matched {TISSUE_PATTERN!r}. Look at the cell-type list above and set "
                   f"TISSUE_PATTERN in Cell 1 to something that appears there.")
counts = meta.groupby(["tissue", "window"]).size()
missing = [k for k in [("salivary gland", "early (stages 10-12)"), ("salivary gland", "late (stages 13-16)")]
           if counts.get(k, 0) < 25]
if missing:
    raise StopHere(f"Too few salivary gland cells in {missing}. The comparison needs both windows. "
                   f"Check the stage values printed above.")
meta = meta[meta.window.notna()].copy()
keep_ids = dict(zip(meta[id_col].astype(str), meta.tissue + " | " + meta.window))
print(f"\n{len(keep_ids):,} cells assigned to a tissue and a window.")

In [ ]:
#@title Cell 4 - Stream the matrix into pseudobulk (a few minutes; the full matrix is never held in memory)
rows = [l.split("\t")[0].strip() for l in open(ROWS) if l.strip()]          # genes
cols = [l.split("\t")[0].strip() for l in open(COLS) if l.strip()]          # cells
print(f"Matrix index: {len(rows):,} rows x {len(cols):,} columns")

with open(MTX) as h:
    header = h.readline()
    line = h.readline()
    while line.startswith("%"):
        line = h.readline()
    n1, n2, nnz = (int(x) for x in line.split())
print(f"Matrix header says {n1:,} x {n2:,} with {nnz:,} non-zero entries")
# EBI writes genes as rows, but check rather than assume
if (n1, n2) == (len(rows), len(cols)):
    gene_axis, cell_axis = 0, 1
elif (n1, n2) == (len(cols), len(rows)):
    gene_axis, cell_axis = 1, 0
    print("   matrix is transposed relative to the index files; handled")
else:
    raise StopHere(f"Matrix is {n1} x {n2} but the index files have {len(rows)} genes and {len(cols)} cells. "
                   "These files do not belong together.")

groups = sorted(set(keep_ids.values()))
gidx = {g: i for i, g in enumerate(groups)}
col_group = np.full(len(cols), -1, dtype=np.int16)
for j, c in enumerate(cols):
    g = keep_ids.get(c)
    if g is not None:
        col_group[j] = gidx[g]
print(f"Columns matched to the annotation table: {int((col_group >= 0).sum()):,} of {len(cols):,}")
if (col_group >= 0).sum() < 0.2 * len(cols):
    raise StopHere("Fewer than a fifth of the matrix columns matched the annotation table, so the cell "
                   "identifiers do not line up. Send me the first few of each and I will fix the matching.")

sums = np.zeros((len(rows), len(groups)), dtype=np.float64)
per_cell = np.zeros(len(cols), dtype=np.float64)
read = 0
with open(MTX) as h:
    h.readline()
    line = h.readline()
    while line.startswith("%"):
        line = h.readline()
    for line in h:                                   # one non-zero entry per line: row col value
        a, b, v = line.split()
        i, j = int(a) - 1, int(b) - 1
        if gene_axis == 1:
            i, j = j, i
        g = col_group[j]
        val = float(v)
        per_cell[j] += val
        if g >= 0:
            sums[i, g] += val
        read += 1
        if read % 20_000_000 == 0:
            print(f"   {read:,} entries read")
print(f"Finished: {read:,} entries")

pb = pd.DataFrame(sums, index=rows, columns=groups)
pb = pb.groupby(level=0).sum()
n_cells = pd.Series(col_group[col_group >= 0]).map({i: g for g, i in gidx.items()}).value_counts().reindex(groups)
print("\nCells and total counts per group:")
display(pd.DataFrame({"cells": n_cells, "counts": pb.sum().round(0)}))
low = n_cells[n_cells < 25]
if len(low):
    raise StopHere(f"These groups have too few cells to pseudobulk: {list(low.index)}.")

# median-of-ratios, so that a few very high genes in one group cannot set the scale
pb = pb[(pb / pb.sum() * 1e6 >= 5).all(axis=1)]
lg = np.log(pb.replace(0, np.nan)); ref = lg.mean(axis=1)
size = np.exp(lg[ref.notna()].sub(ref[ref.notna()], axis=0).median())
L = np.log2(pb / size / pb.sum().mean() * 1e6 + 1)
print(f"\nGenes expressed in every group: {len(L):,}")
print("Size factors:", dict(size.round(2)))

In [ ]:
#@title Cell 5 - Gene symbols, the two contrasts, and the positive control
if not os.path.exists(GTF_PATH):
    raise StopHere(f"Gene annotation not found at {GTF_PATH}. It is the same file Route B and Route D use; "
                   "fix GTF_PATH in Cell 1.")
id2sym = {}
with gzip.open(GTF_PATH, "rt") as h:
    for line in h:
        if "\tgene\t" not in line:
            continue
        gid = re.search(r'gene_id "([^"]+)"', line); gnm = re.search(r'gene_name "([^"]+)"', line)
        if gid and gnm:
            id2sym[gid.group(1)] = gnm.group(1)
L.index = [id2sym.get(re.sub(r"\.\d+$", "", i), i) for i in L.index]
L = L.groupby(level=0).sum()
print(f"Genes after converting to symbols: {len(L):,}")

SG, OTH = "salivary gland", "other embryonic cells"
EARLY, LATE = "early (stages 10-12)", "late (stages 13-16)"
need = [f"{t} | {w}" for t in (SG, OTH) for w in (EARLY, LATE)]
miss = [c for c in need if c not in L.columns]
if miss:
    raise StopHere(f"Missing pseudobulk groups: {miss}. Available: {list(L.columns)}")

fc_sg = L[f"{SG} | {LATE}"] - L[f"{SG} | {EARLY}"]          # change across the window, in salivary gland
fc_oth = L[f"{OTH} | {LATE}"] - L[f"{OTH} | {EARLY}"]       # the same change in every other cell type
fc = fc_sg - fc_oth                                          # what is specific to the salivary gland
print("\nThree contrasts are available:")
print("   fc_sg   late vs early, salivary gland only")
print("   fc_oth  late vs early, all other cells - the embryo-wide maturation background")
print("   fc      fc_sg minus fc_oth - the SALIVARY-GLAND-SPECIFIC change, used for the screens below")

ALIASES = {"geminin": ["Gem", "CG3183"], "dup": ["Cdt1"], "Zw": ["G6PD", "G6pd"], "Ero1L": ["Ero1", "CG1333"],
           "PHGPx": ["CG12013"], "Gss1": ["Gss"], "Gr1": ["Gr", "GR"], "Trxr-1": ["Trxr1"], "Jafrac1": ["Prx2"],
           "Su(var)3-3": ["dLsd1", "Lsd1", "Su-var(3)3", "Suvar(3)3", "CG17149"], "Rrp1": ["APEX1", "CG3178"],
           "Sec61alpha": ["Sec61"], "SrpRbeta": ["SrpRb"]}


def resolve(g, present):
    if g in present:
        return g
    for a in ALIASES.get(g, []):
        if a in present:
            return a
    return None


PRESENT = set(L.index)
CONTROLS = {
    "POSITIVE - CrebA secretory pathway targets (must RISE in the gland)":
        "Sec61beta Sec61alpha SrpRalpha SrpRbeta Spase25 p24-1 zetaCOP Sec23 Sec31 Srp54 Sar1",
    "POSITIVE - salivary gland identity (should stay on)": "fkh sage CrebA Pasilla hth",
    "NEGATIVE - mitotic genes (gland leaves the mitotic cycle early)": "CycB polo aurB stg Cdk1 CycA pav tum",
}
print("\n=== Positive control: does the gland build its secretory machinery across these windows? ===")
for name, gs in CONTROLS.items():
    g = [r for x in gs.split() if (r := resolve(x, PRESENT))]
    if len(g) < 3:
        print(f"{name}: only {len(g)} genes found, skipped"); continue
    print(f"\n{name}  ({len(g)} genes: {', '.join(g)})")
    print(f"   salivary gland, late vs early: {fc_sg.loc[g].median():+.2f}")
    print(f"   other cells, late vs early:    {fc_oth.loc[g].median():+.2f}")
    print(f"   gland-specific:                {fc.loc[g].median():+.2f}")
crebA = [r for x in CONTROLS[list(CONTROLS)[0]].split() if (r := resolve(x, PRESENT))]
if len(crebA) >= 3 and fc_sg.loc[crebA].median() <= 0:
    print("\nWARNING: the CrebA secretory targets do not rise in the gland across these windows. That is the one "
          "thing this comparison must show, so the staging or the cell-type annotation is probably wrong. "
          "Treat everything below as uninterpretable until this is resolved.")

In [ ]:
#@title Cell 6 - Redox programmes: what changes in the gland between the two windows
PROGRAMMES = {
    "REDOX: glutathione S-transferases": "GstD1 GstD2 GstD3 GstD5 GstD9 GstE1 GstE3 GstE6 GstE7 GstE9 GstE12 GstE14 GstS1 GstO1 GstO2 GstO3 GstT1 GstZ1 GstZ2 Mgstl",
    "REDOX: glutathione synthesis & recycling": "Gclc Gclm Gss1 Gss2 Gr1 Grx1 Grx1t PHGPx GILT1",
    "REDOX: thioredoxin / peroxiredoxin": "Trxr-1 Trxr-2 Trx-2 TrxT dhd Jafrac1 Jafrac2 Prx3 Prx5 Prx2540-1 Prx2540-2 Srx",
    "REDOX: SOD and catalase": "Sod1 Sod2 Sod3 Cat",
    "REDOX: NADPH supply": "Zw Pgd Men Men-b Idh Idh3a Nmdmc Tkt Taldo",
    "REDOX: cnc (NRF2) targets": "cnc Keap1 GstD1 Gclc Gclm Cyp6a2 Cyp6a8 Ugt35a Jafrac1 Prx3",
    "REDOX: H2O2-producing enzymes": "Ero1L Duox Nox Aox1 Uro Cyp4e3",
    "REDOX: mitochondrial respiration": "ND-75 ND-49 ND-30 ND-42 SdhA SdhB UQCR-C1 UQCR-C2 Cyt-c-p COX4 COX5A ATPsynbeta blw",
    "EXPLORE: ER folding load": "Xbp1 Hsc70-3 Ire1 PEK crc Pdi ERp60 Calr Ero1L Sec61alpha Sec61beta Srp54",
    "EXPLORE: ribosome / protein synthesis": "RpL3 RpL4 RpL5 RpL7 RpL8 RpL10 RpL11 RpL13 RpS2 RpS3 RpS4 RpS6 RpS8 RpS9 RpS11 RpS15",
    "EXPLORE: licensing and re-replication": "geminin dup Cdc6 Orc1 Mcm2 Mcm3 dpa Mcm5 Mcm6 Mcm7 CycE E2f1 Myc",
    "EXPLORE: p53 / DNA damage": "p53 Ku80 Irbp lok mei-41 tefu rad50 Xrp1 spn-A mus304",
    "CONTROL: heat shock": "Hsp70Aa Hsp83 Hsp68 Hsp27 Hsp26 Hsp23 DnaJ-1 Hsf",
    "CONTROL: immediate-early genes": "Hr38 sr kay Jra puc",
    "CONTROL: housekeeping": "Act5C alphaTub84B RpL32 Gapdh1 Tbp eIF4A Pgk Ef1alpha48D",
}
_res, _renamed = {}, {}
for n, v in PROGRAMMES.items():
    out = []
    for g in v.split():
        r = resolve(g, PRESENT)
        if r:
            out.append(r)
            if r != g:
                _renamed[g] = r
    _res[n] = out
PROGRAMMES = {n: v for n, v in _res.items() if len(v) >= 3}
if _renamed:
    print("Found under a synonym:", ", ".join(f"{k} -> {v}" for k, v in _renamed.items()))
for n, v in PROGRAMMES.items():
    print(f"{n}: {len(v)} genes")

level = L.mean(axis=1)
bins = pd.qcut(level.rank(method="first"), 10, labels=False)
avoid = {g for v in PROGRAMMES.values() for g in v}
pools = {b: [g for g in bins.index[bins == b] if g not in avoid] for b in range(10)}
controls = [n for n in PROGRAMMES if n.startswith("CONTROL")]
rows = []
for label, series in [("gland-specific (gland minus other cells)", fc),
                      ("salivary gland, late vs early", fc_sg),
                      ("other cells, late vs early", fc_oth)]:
    for n, gs in PROGRAMMES.items():
        obs = series.loc[gs].median()
        null = np.array([series.loc[[rng.choice(pools[bins[g]]) for g in gs]].median() for _ in range(N_RANDOM_SETS)])
        c = np.median(null)
        p = (1 + np.sum(np.abs(null - c) >= abs(obs - c))) / (N_RANDOM_SETS + 1)
        beyond = all(np.sign(obs - series.loc[PROGRAMMES[k]].median()) == np.sign(obs - c) for k in controls)
        rows.append({"contrast": label, "programme": n, "genes": len(gs), "log2_change": obs,
                     "random_median": c, "p_vs_random": p, "beyond_all_controls": beyond,
                     "pct_of_all_genes": series.rank(pct=True).loc[gs].median() * 100})
res = pd.DataFrame(rows)
for ct in res.contrast.unique():
    m = (res.contrast == ct) & ~res.programme.str.startswith("CONTROL")
    p = res.loc[m, "p_vs_random"].to_numpy(); o = np.argsort(p)
    q = np.minimum.accumulate((p[o] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]
    qq = np.empty_like(p); qq[o] = np.minimum(q, 1); res.loc[m, "q_fdr"] = qq
display(res[res.contrast == res.contrast.iloc[0]].round(3))
print("\nThe gland-specific contrast is the one that matters: it removes changes happening everywhere in the "
      "embryo between these windows. The other two are shown in the saved table for reference.")

In [ ]:
#@title Cell 7 - Compartment-resolved screen, gene table and saving
COMPART = {
    ("nucleus", "H2O2 source (chromatin demethylation)"): "Su(var)3-3",
    ("nucleus", "thioredoxins and oxidative DNA repair"): "dhd TrxT Rrp1 Ogg1",
    ("ER", "H2O2 source"): "Ero1L",
    ("ER", "H2O2 removal"): "Jafrac2",
    ("ER", "folding load"): "Hsc70-3 Pdi ERp60 Calr Sec61alpha Sec61beta Xbp1 Ire1 PEK crc Srp54",
    ("cytosol", "H2O2 removal"): "Jafrac1 Prx2540-1 Prx2540-2 PHGPx",
    ("cytosol", "thioredoxin system"): "Trxr-1 Trx-2 Srx",
    ("cytosol", "glutathione supply"): "Gclc Gclm Gss1 Gss2 Gr1 Grx1 Grx1t",
    ("cytosol", "superoxide dismutase"): "Sod1 Sod3",
    ("mitochondria", "H2O2 removal"): "Prx3 Prx5 Trxr-2",
    ("mitochondria", "superoxide dismutase"): "Sod2",
    ("peroxisome", "H2O2 removal"): "Cat",
    ("plasma membrane", "H2O2 source"): "Duox Nox",
}
resolved, missing = {}, []
for (comp, role), v in COMPART.items():
    keep = []
    for g in v.split():
        r = resolve(g, PRESENT)
        keep.append(r) if r else missing.append(g)
    resolved[(comp, role)] = keep
if missing:
    print("Not detected in this dataset:", ", ".join(missing))

pct = fc.rank(pct=True)
rows, crows = [], []
for (comp, role), gs in resolved.items():
    for g in gs:
        rows.append({"gene": g, "compartment": comp, "role": role,
                     "gland_specific": float(fc.loc[g]), "pct_of_all_genes": float(pct.loc[g]) * 100,
                     "gland_late_vs_early": float(fc_sg.loc[g]), "other_late_vs_early": float(fc_oth.loc[g])})
    if not gs:
        continue
    obs = float(fc.loc[gs].median())
    null = np.array([fc.loc[[rng.choice(pools[bins[g]]) for g in gs]].median() for _ in range(N_RANDOM_SETS)])
    c = float(np.median(null))
    crows.append({"compartment": comp, "role": role, "genes": len(gs), "gland_specific": obs,
                  "random_median": c, "vs_random": obs - c,
                  "p_vs_random": ((1 + np.sum(np.abs(null - c) >= abs(obs - c))) / (N_RANDOM_SETS + 1)
                                  if len(gs) >= 3 else np.nan),
                  "median_pct": float(pct.loc[gs].median()) * 100})
gene_tab = pd.DataFrame(rows).set_index("gene")
comp_tab = pd.DataFrame(crows).sort_values(["compartment", "role"])
print("\nEvery redox gene by where it works. Positive = rises in the gland beyond the embryo-wide change.")
display(gene_tab.sort_values(["compartment", "role", "gland_specific"]).round(2))
display(comp_tab.round(3))

word = lambda p: "up" if p > 65 else "down" if p < 35 else "flat"
print("\n--- Source against sink, within each compartment (gland-specific) ---")
for comp in ["ER", "nucleus", "cytosol", "mitochondria"]:
    src = [g for (c, ro), gs in resolved.items() if c == comp and "source" in ro for g in gs]
    snk = [g for (c, ro), gs in resolved.items() if c == comp and ("removal" in ro or "thioredoxin" in ro) for g in gs]
    if not src or not snk:
        print(f"  {comp}: not enough detected genes on both sides ({len(src)} source, {len(snk)} removal)")
        continue
    s, k = float(fc.loc[src].median()), float(fc.loc[snk].median())
    sp, kp = float(pct.loc[src].median()) * 100, float(pct.loc[snk].median()) * 100
    print(f"  {comp}: sources {s:+.2f} ({sp:.0f}th pct, {word(sp)}) [{', '.join(src)}]; "
          f"removal {k:+.2f} ({kp:.0f}th pct, {word(kp)}) [{', '.join(snk)}]")
    if word(sp) == "up" and word(kp) != "up":
        print(f"      Production rises and removal does not follow: an imbalance local to the {comp}.")
    elif word(sp) == "up" and word(kp) == "up":
        print("      Both rise together, so capacity is tracking production.")
    elif word(sp) != "up" and word(kp) == "down":
        print(f"      Removal falls without a rise in production: less reducing capacity in the {comp}.")
    else:
        print("      Neither side moves clearly against the genome.")

fig, ax = plt.subplots(figsize=(8, 0.34 * len(comp_tab) + 1.4))
lab = [f"{c} - {r}" for c, r in zip(comp_tab.compartment, comp_tab.role)]
ax.barh(lab, comp_tab.vs_random, color=["#b2432f" if v < 0 else "#2f6db2" for v in comp_tab.vs_random])
ax.axvline(0, color="black", lw=0.8); ax.invert_yaxis()
ax.set_xlabel("gland-specific log2 change, minus random genes")
ax.set_title(f"{GSE}: salivary gland, stages 13-16 vs 10-12", loc="left", fontsize=10)
plt.tight_layout(); plt.savefig(os.path.join(GDIR, "chart_compartments.png"), dpi=150); plt.show()

res.to_csv(os.path.join(GDIR, "programmes.csv"), index=False)
gene_tab.to_csv(os.path.join(GDIR, "compartments_genes.csv"))
comp_tab.to_csv(os.path.join(GDIR, "compartments_summary.csv"), index=False)
L.to_csv(os.path.join(GDIR, "pseudobulk_log2cpm.csv"))
print("\nSaved to", GDIR)
print("""
Reading this. The gland-specific contrast already removes anything happening across the whole embryo between
these windows, which is the main confound. Two things it cannot remove: the gland is differentiating into a
secretory tissue AND entering endocycles over the same hours, so this comparison cannot separate those, exactly
as the larval salivary gland could not. What it can do is date the change - whether the ER folding load and any
redox shift are already underway in the early window or only appear in the late one.
""")